# CLM Vision: asking about X-rays

A research prototype, **not a clinical tool**. This notebook sends radiographs from
[FracAtlas](https://figshare.com/articles/dataset/The_dataset/22363012) to a CLM **vision head** served by
`clm-serve`, and asks "Is there a bone fracture?" as a typed question.

![CLM vision architecture](../assets/clm_vision_arch.svg)

The state tower is new: a frozen image encoder (MedSigLIP-448 or Qwen3-VL-4B) and a trained `state_head`.
The action tower is the released CLM one: the option texts go through Qwen3-8B (served by vLLM) and the
released `action_head`.

![FracAtlas test AUROC](../assets/fracatlas_auroc.svg)

| on the FracAtlas test split (600 images, 109 fractured) | AUROC | sens / spec at val threshold | balanced acc. |
|---|---:|---:|---:|
| CLM head · MedSigLIP-448 | 0.898 ± 0.012 | 0.810 / 0.838 | 0.824 |
| linear probe · MedSigLIP-448 | 0.898 | 0.771 / 0.859 | 0.815 |
| CLM head · Qwen3-VL-4B | 0.887 ± 0.012 | 0.820 / 0.778 | 0.799 |
| linear probe · Qwen3-VL-4B | 0.890 | 0.817 / 0.849 | 0.833 |

All statistics, ablations and caveats are in the README section
"Vision: fracture detection on X-rays" and in `results/fracatlas.md`.

## Before you run it

Run these from the repo root, in a terminal.

**1. Data and a trained head.** Run the pipeline in the README (steps 1 to 4), or at least:

```bash
python tools/fracatlas_download.py
python train/embed_images.py --encoder google/medsiglip-448
python train/embed_options.py
python train/sweep_vision.py --encoder medsiglip
best=$(python -c "import json; print(json.load(open('runs/fracatlas/sweep/medsiglip/sweep.json'))['best']['dir'])")
python train/calibrate_vision.py --ckpt "$best/s0/best_head.pt" --out checkpoints/fracatlas-medsiglip.pt
```

The last step calibrates the head on val (Platt scaling and a decision threshold), so the served
probabilities are meaningful and `choice` catches most fractures.

**2. The Qwen3-8B encoder (vLLM) on a GPU machine**, port 8090:

```bash
vllm serve Qwen/Qwen3-8B --served-model-name qwen3-8b --runner pooling --max-model-len 2048 \
    --gpu-memory-utilization 0.75 --port 8090
```

On a Mac, run vLLM on a GPU box and forward the port: `ssh -N -L 8090:localhost:8090 <gpu-box>`.
Or, for a quick local test without vLLM, serve the cached option embeddings instead:
`python tools/options_embedder.py` (it answers only the head's option texts).

**3. `clm-serve` with the vision head** (the image encoder runs inside it, on CUDA or the Mac GPU):

```bash
clm-serve --model fracatlas-medsiglip=checkpoints/fracatlas-medsiglip.pt
```

**4. Jupyter**: `pip install jupyter`, then open this notebook from the `examples/` folder.

## Setup

In [ ]:
import base64, csv, json, random
from pathlib import Path

import numpy as np
import torch
from IPython.display import HTML, display
from PIL import Image

from clm import CLMClient, CLMError, Choice, ImageState, Noul

REPO = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
DATA = REPO / "data" / "fracatlas"
CKPT = REPO / "checkpoints" / "fracatlas-medsiglip.pt"   # the head clm-serve was started with
MODEL = "fracatlas-medsiglip"                              # its name in clm-serve --model NAME=PATH
BASE_URL = "http://127.0.0.1:8700"

client = CLMClient(BASE_URL)
print("clm-serve up:", client.health())
for m in client.models():
    print(f"  {m['name']:<22} {m['description']}")

## The question the head was trained on

A vision head only knows the option texts it was trained against, so read them from the checkpoint.
The question's `instructions` do not reach the image encoder: the image is always embedded with the
prompt used in training.

In [ ]:
cfg = torch.load(CKPT, map_location="cpu", weights_only=False)["cfg"]
print("state encoder :", cfg["state_encoder"], "| modality:", cfg["state_modality"])
print("image prompt  :", cfg["state_embedding"].get("instruction", "(none, pooled image embedding)"))
print("options       :", json.dumps(cfg["options"], indent=2))
for c in cfg.get("calibration", []):
    print(f"calibration   : {c['form']:<6} threshold {c['threshold']:.3f} ({c['rule']}); val sens/spec "
          f"{c['val']['sens_spec_raw_at_0.5'][0]:.2f}/{c['val']['sens_spec_raw_at_0.5'][1]:.2f} raw -> "
          f"{c['val']['sens_spec_at_threshold'][0]:.2f}/{c['val']['sens_spec_at_threshold'][1]:.2f} calibrated")
if not cfg.get("calibration"):
    print("calibration   : none. Run train/calibrate_vision.py, or choice will miss many fractures")

questions = {
    "fx": Choice(instructions=cfg["instructions"], criteria=cfg["options"]),
    "fx_noul": Noul(instructions=cfg["instructions"],
                    criteria={"true": cfg["options"]["fracture"], "false": cfg["options"]["no_fracture"]}),
}

## Ask about a few radiographs

Images come from the **validation** split, so the test split stays untouched.

In [ ]:
val = list(csv.DictReader(open(DATA / "splits" / "val.csv")))
rng = random.Random(0)
sample = rng.sample([r for r in val if r["fractured"] == "1"], 3) + rng.sample([r for r in val if r["fractured"] == "0"], 3)

def ask(row):
    r = client.system_one(ImageState(path=str(DATA / "raw" / row["path"])), questions, model=MODEL)
    a = r.answers["fx"]
    return a.probabilities["fracture"], a.choice, a.threshold, r.latency_ms

for row in sample:
    p, choice, thr, ms = ask(row)
    im = Image.open(DATA / "raw" / row["path"]).convert("L")
    im.thumbnail((220, 220))
    display(im)
    display(HTML(f"<b>{row['image_id']}</b> · {row['region']} · {row['view']} · truth: "
                 f"<b>{'fracture' if row['fractured'] == '1' else 'no fracture'}</b> · "
                 f"answer: <b>{choice}</b> · p(fracture) = {p:.3f}"
                 + (f" (threshold {thr:.3f})" if thr is not None else "") + f" · {ms:.0f} ms"))

## Sanity check on val

This scores val images through the server and compares them with what was recorded offline: the AUROC
from training, and the sensitivity and specificity the calibration step measured at its threshold.
`choice` already applies that threshold. Without calibration, the raw head is overconfident (most
probabilities near 0 or 1) and its plain `choice` misses a third or more of fractures.

`N = None` scores all 609 val images (about 2 minutes with MedSigLIP).

In [ ]:
from sklearn.metrics import roc_auc_score

N = 200
rows = val if N is None else random.Random(1).sample(val, N)
y = np.array([int(r["fractured"]) for r in rows])
out = [ask(r) for r in rows]
p = np.array([o[0] for o in out])
said_fracture = np.array([o[1] == "fracture" for o in out])

ck = torch.load(CKPT, map_location="cpu", weights_only=False)
print(f"val AUROC through clm-serve : {roc_auc_score(y, p):.3f}  ({len(y)} images, {y.sum()} fractured)")
print(f"val AUROC at training time  : {ck['metrics']['auroc']:.3f}  (all 609 val images)")
print(f"probabilities < 0.01 or > 0.99: {np.mean((p < 0.01) | (p > 0.99)):.0%}")
print(f"served choice: sensitivity {said_fracture[y == 1].mean():.2f}, specificity {(~said_fracture[y == 0]).mean():.2f}")
cal = next((c for c in ck["cfg"].get("calibration", []) if c["form"] == "choice"), None)
if cal:
    print(f"calibration record (all val): sensitivity {cal['val']['sens_spec_at_threshold'][0]:.2f}, "
          f"specificity {cal['val']['sens_spec_at_threshold'][1]:.2f} at p >= {cal['threshold']:.3f}")

## The same request over plain HTTP

The wire form of an image state is `{"type": "image", "image": "<base64>"}`; `ImageState(path=...)`
does this for you on the client side. The server never reads file paths.

In [ ]:
import requests

path = DATA / "raw" / sample[0]["path"]
body = {"model": MODEL,
        "state": {"type": "image", "image": base64.b64encode(path.read_bytes()).decode()},
        "questions": {"fx": {"type": "choice", "instructions": cfg["instructions"], "criteria": cfg["options"]}}}
r = requests.post(f"{BASE_URL}/v1/systemone", json=body, timeout=120)
print(r.status_code, r.headers.get("X-CLM-Latency-Ms"), "ms")
print(json.dumps(r.json()["answers"], indent=2))

## Guardrails

A vision head refuses text states, a text head refuses images, and the server refuses file paths.

In [ ]:
def try_call(label, fn):
    try:
        fn()
        print(f"{label}: accepted (unexpected)")
    except CLMError as e:
        print(f"{label}: {e}")

try_call("text state  -> vision head", lambda: client.system_one("X-ray of a wrist", questions, model=MODEL))
try_call("image state -> clm-latest ", lambda: client.system_one(ImageState(path=str(path)), questions, model="clm-latest"))
try_call("file path over HTTP       ", lambda: client.system_one({"type": "image", "path": str(path)}, questions, model=MODEL))

## Optional: in-process, without clm-serve

`Engine` does the same routing in-process; it needs only the vLLM endpoint for the option texts.
It loads its own copy of the image encoder, so it is off by default.

In [ ]:
RUN_IN_PROCESS = False

if RUN_IN_PROCESS:
    from clm import Engine
    engine = Engine(emb_url="http://127.0.0.1:8090/v1/embeddings", models={MODEL: str(CKPT)})
    out = engine.answer(ImageState(path=str(path)), questions, model=MODEL)
    print(json.dumps(out["answers"], indent=2))